# Example 1.1.1 / 2: Flat Sets

Companion notebook for **1.1.1 Flat vs Indexed Sets** in the AMPL lab Documentation, section "Flat Sets". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Flat Sets

`within` makes AMPL check the data: a member of `B` that isn't a product is rejected. amplpy
accepts the assignment, but AMPL checks the members when the set is first used, here when
`to_list()` reads it, and amplpy raises a `RuntimeError`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;            # products
set B within PROD;   # subset of products; the data below breaks it

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["B"] = ["bands", "rods"]

In [5]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    print(ampl.set["B"].to_list())
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "display" command:
error processing set B:
	B contains rods, which is not
	within PROD;


### Pure AMPL: data.dat and model.run

Pure AMPL; the error appears when `display` uses `B`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set B := bands rods;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display B;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing set B:
		B contains rods, which is not
		within PROD;
